# OpenAI API 첫걸음 — 가장 단순한 호출부터

이 노트북은 **LLM 에게 말을 거는 가장 기본적인 방법**을 다룹니다.
LangChain 같은 프레임워크 없이 OpenAI 공식 라이브러리만 씁니다.
나중에 배울 모든 것이 결국 이 호출 위에 얹힙니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리와 API 키 |
| **Step 3~5** | 첫 대화 — `messages` 를 만들어 보내고 답 받기 |
| **Step 6~9** | 역할 주기 — `system` 으로 성격을 정하고 다시 묻기 |
| **Step 10~11** | 응답 뜯어보기 — 무엇이 들어 있나 |

> **이 노트북에서 기억할 한 가지**
> LLM 호출은 `messages` 라는 **목록 하나를 보내고 답을 받는 일**이 전부입니다.
> 복잡해 보이는 에이전트도 결국 이 목록을 잘 만들어 주는 장치입니다.

## Step 1. 준비물 세 가지

| 가져오는 것 | 하는 일 |
|---|---|
| `os` | 환경변수를 읽는다 |
| `dotenv` | `.env` 파일을 환경변수로 올려 준다 |
| `OpenAI` | OpenAI 공식 클라이언트 |

### 왜 `.env` 를 쓰나

API 키를 코드에 직접 적으면 **git 에 그대로 올라갑니다.**
한 번 올라간 키는 나중에 지워도 커밋 기록에 남습니다.

그래서 키는 `.env` 파일에 두고, 그 파일은 `.gitignore` 로 제외합니다.
코드에는 "환경변수에서 읽어라"만 적습니다.

```
llm_edu/.env        <-  OPENAI_API_KEY=sk-...   (git 에 안 올라감)
```

In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. API 키가 제대로 들어왔는지 확인

```python
load_dotenv(override=True)
```

`.env` 파일을 읽어 환경변수로 올립니다.
`override=True` 는 **이미 같은 이름의 환경변수가 있어도 `.env` 값으로 덮어쓰라**는 뜻입니다.
예전에 터미널에 설정해 둔 낡은 키가 섞이는 사고를 막아 줍니다.

### 앞 8자만 찍는 이유

```python
print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
```

키 전체를 출력하면 **노트북 결과창에 그대로 저장**되고, 그 노트북을 공유하는 순간 키가 샙니다.
있는지 없는지만 확인하면 되므로 앞부분만 봅니다.

> **확인을 먼저 하는 이유**: 나중에 호출이 실패했을 때
> 원인이 '키 문제'인지 '코드 문제'인지 바로 가를 수 있습니다.

In [22]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


## Step 3. `messages` — 대화의 기본 단위

LLM 에 보내는 것은 문자열이 아니라 **역할이 붙은 메시지들의 목록**입니다.

```python
[{"role": "user", "content": "안녕하세요..."}]
```

### 역할(role) 세 가지

| role | 누가 한 말인가 | 쓰임 |
|---|---|---|
| `system` | 개발자가 모델에게 주는 지시 | 성격·말투·규칙을 정한다 |
| `user` | 사용자가 한 말 | 실제 질문 |
| `assistant` | 모델이 한 말 | 이전 답변을 다시 넣어 **대화를 이어갈 때** 쓴다 |

지금은 가장 단순하게 `user` 하나만 넣습니다.
`messages` 를 마지막 줄에 그냥 적어 **어떤 모양인지 눈으로 확인**하고 갑니다.

In [23]:
message="안녕하세요, 만나뵈서 반갑습니다. 내 이름은 홍길동입니다."
messages = [{"role": "user", "content": message}]
messages

[{'role': 'user', 'content': '안녕하세요, 만나뵈서 반갑습니다. 내 이름은 홍길동입니다.'}]

## Step 4. 클라이언트 만들기

```python
openai = OpenAI()
```

괄호 안이 비어 있는데도 동작합니다.
`OpenAI()` 가 **환경변수 `OPENAI_API_KEY` 를 알아서 찾아 쓰기** 때문입니다.
Step 2 에서 `.env` 를 올려 두었으므로 따로 넘겨줄 필요가 없습니다.

> 키를 직접 넘기고 싶다면 `OpenAI(api_key=...)` 도 됩니다.
> 하지만 그러면 키가 코드에 적히므로 권하지 않습니다.

In [24]:
# OpenAI 크레딧이 없어서 로컬 Ollama 로 보냅니다.
# 크레딧을 충전하면 아래 줄을 openai = OpenAI() 로 바꾸고 model 을 gpt-5-nano 등으로 되돌리세요.
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

## Step 5. 첫 호출 — 보내고 받기

```python
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content
```

필요한 것은 **모델 이름과 `messages` 둘뿐**입니다.

### `choices[0]` 이 왜 붙나

응답의 `choices` 는 **목록**입니다. 기본값은 답을 하나만 받으므로 항상 `[0]` 입니다.
`n=3` 처럼 주면 같은 질문에 서로 다른 답을 3개 받을 수 있고, 그때 `choices` 가 길어집니다.

```
response
  └ choices[0]
      └ message
          └ content   <-  우리가 원하는 답변 글자
```

모델이 "홍길동님"이라고 이름을 불러 줍니다. 우리가 보낸 문장 안에 있던 정보입니다.

In [25]:
response = openai.chat.completions.create(model="qwen2.5:3b", messages=messages)
response.choices[0].message.content

'안녕하세요 홍길동先生, 반갑습니다. 저는 인공지능으로, 다양한 정보를 제공하고 도와드릴 수 있습니다. 어떤 도움이 필요하신가요?'

## Step 6. `system` 프롬프트 — 모델에게 역할을 준다

여기서부터가 진짜입니다. 같은 질문이라도 **역할을 어떻게 주느냐에 따라 답이 완전히 달라집니다.**

```python
system_prompt = """
당신은 IT 보안 전문가입니다. ...
"""
```

`system` 은 사용자에게 보이지 않는 **무대 뒤 지시문**입니다. 보통 이런 것을 적습니다.

| 적는 것 | 예 |
|---|---|
| 역할 | "당신은 IT 보안 전문가입니다" |
| 범위 | "보안 외의 질문에는 답하지 마세요" |
| 말투 | "초보자도 이해할 수 있게 설명하세요" |
| 형식 | "항상 번호 매긴 목록으로 답하세요" |

> 뒤에서 배울 RAG·에이전트도 결국 이 `system` 에
> **무엇을 어떻게 적을 것인가**의 싸움입니다.

In [26]:
system_prompt = """
당신은 IT 보안 전문가입니다. 
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
반드시 한국어(한글)로만 답하고, 한자·중국어·영어 등 다른 언어는 섞지 마세요.
"""

## Step 7. `user` 프롬프트 — 실제 질문

```python
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?
"""
```

질문만 던지지 않고 **"나는 보안 책임자다"라는 상황을 함께 줬습니다.**
같은 질문도 묻는 사람이 누구냐에 따라 적절한 답이 달라지기 때문입니다.

| 질문자 | 기대하는 답 |
|---|---|
| 보안 책임자 | 공급망 공격, 제로데이 같은 **대응 전략** |
| 일반 직원 | 비밀번호 관리 같은 **생활 수칙** |

**상황을 적어 주는 것만으로 답의 수준이 달라집니다.**

In [27]:
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?
"""

## Step 8. 두 역할을 하나의 목록으로

```python
messages = [{"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}]
```

`system` 을 먼저, `user` 를 뒤에 놓습니다. 순서가 곧 **모델이 읽는 순서**입니다.

> **주의**: 여기서 `messages` 를 **새로 덮어썼습니다.**
> Step 3 의 "홍길동" 메시지는 이제 목록에 없습니다.
>
> 그래서 모델은 **홍길동이 누구였는지 기억하지 못합니다.**
> LLM 호출은 매번 완전히 독립적이고, 우리가 보낸 `messages` 가 전부입니다.
> 대화를 이어가려면 지난 대화를 **우리가 직접 목록에 쌓아 보내야** 합니다.
> 이 이야기는 뒤의 `1_6_stateless` 노트북에서 자세히 다룹니다.

In [28]:
messages = [ {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

## Step 9. 역할을 준 상태로 호출

```python
response = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)
```

Step 5 와 **똑같은 함수**입니다. 달라진 것은 두 가지뿐입니다.

| | Step 5 | Step 9 |
|---|---|---|
| `messages` | `user` 1개 | `system` + `user` |
| `model` | `gpt-5-nano` | `gpt-4.1-mini` |

모델 이름 문자열만 바꾸면 다른 모델로 갈아탈 수 있습니다.
**코드는 그대로입니다.** 작고 싼 모델로 개발하고 필요할 때 큰 모델로 올리는 식으로 씁니다.

In [29]:
response = openai.chat.completions.create(
    model = "qwen2.5:3b",
    messages = messages
)

## Step 10. 응답에 무엇이 들어 있나

보통은 답변 글자만 꺼내 쓰지만, 한 번쯤 **전체를 펼쳐 보는 것**이 좋습니다.

```python
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))
```

`ensure_ascii=False` 를 빼면 한글이 `\uc548\ub155` 처럼 깨져 보입니다.

### 눈여겨볼 항목

| 항목 | 뜻 |
|---|---|
| `choices[0].message.content` | 답변 본문 — 평소 쓰는 것 |
| `choices[0].finish_reason` | 왜 멈췄나. `stop` 은 정상 종료, `length` 는 **글자 제한에 걸려 잘림** |
| `usage.prompt_tokens` | 보낸 토큰 수 |
| `usage.completion_tokens` | 받은 토큰 수 |
| `model` | 실제로 응답한 모델 (요청한 이름과 다를 수 있다) |
| `id` | 이 응답의 고유 번호. 문제 생겼을 때 추적용 |

> **`usage` 가 곧 요금입니다.** 토큰 단위로 과금되므로
> 비용이 걱정될 때 가장 먼저 봐야 할 숫자입니다.
> `finish_reason` 이 `length` 로 나온다면 답이 중간에 잘린 것이니
> `max_tokens` 를 늘리거나 질문을 쪼개야 합니다.

In [30]:
import json
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "chatcmpl-867",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "현대의 IT 보안에서 가장 대두하고 있는 이슈로는 다음과 같은 것들이 주목 받고 있습니다:\n\n1. **모바일 보안**: 안심이 안되는 기기 관리, 보안 강화, 사용자 개인정보 보호 등이 지속적인 관심사입니다. 또한, 기기의 안전성을 입증하는 방법에 대한 연구도 지속되고 있습니다.\n\n2. **인터넷 연결 기기의 보안**: 스마트폰, 태블릿, 로그인기기 등 다양한 인터넷 기기의 보안 강화가 중요한 시점입니다. 또한 이러한 기기들이 악용되는 방법을 대처하기 위한 지식도 중요합니다.\n\n3. **신뢰성 높은 접근 통제**: 인벤토리에 관해서, 접근 통제 시스템, 악성 소프트웨어 감지 시스템 등은 중요한 부분입니다. 더 중요한 문제는 접근 통제의 범위를 어디까지 확대로 확장할 것인지 하는 것입니다.\n\n4. **데이터 보안**: 데이터의 유효성, 접근 가능성을 위한 통제, 그리고 데이터의 중요성을 이해하며 보안 전략을 만드는 것입니다.\n\n5. **개인 정보 보호**: 개인 정보가 침해되는 경우 대처 방법, 그리고 개인 정보의 수집, 보유, 공유 방침 등이 중요한 이슈입니다. \n\n6. **그리스포인트 보안**: 그린스포인트는 사용자에게 주도적으로 보안을 알리는 것이 중요하다고 지적함을 상기해야 합니다. \n\n각 회사는 이러한 이슈를 이해하고, 고객과 직원들의 개인정보를 보호하는데 집중해야 합니다. 새로운 기술과 시각 속에 항상 관심을 가져야 합니다.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_

## Step 11. 실무에서 쓰는 건 결국 이 한 줄

```python
print(response.choices[0].message.content)
```

Step 10 에서 본 긴 JSON 중 **우리가 실제로 쓰는 것은 이 한 줄**입니다.
나머지는 문제가 생겼을 때 들여다보는 정보입니다.

Step 5 와 달리 `print()` 로 감쌌습니다.
셀 마지막 줄에 그냥 두면 `'...\n...'` 처럼 줄바꿈이 글자로 보이지만,
`print()` 로 찍으면 **줄바꿈과 목록이 제대로 보입니다.**

In [31]:
print(response.choices[0].message.content)

현대의 IT 보안에서 가장 대두하고 있는 이슈로는 다음과 같은 것들이 주목 받고 있습니다:

1. **모바일 보안**: 안심이 안되는 기기 관리, 보안 강화, 사용자 개인정보 보호 등이 지속적인 관심사입니다. 또한, 기기의 안전성을 입증하는 방법에 대한 연구도 지속되고 있습니다.

2. **인터넷 연결 기기의 보안**: 스마트폰, 태블릿, 로그인기기 등 다양한 인터넷 기기의 보안 강화가 중요한 시점입니다. 또한 이러한 기기들이 악용되는 방법을 대처하기 위한 지식도 중요합니다.

3. **신뢰성 높은 접근 통제**: 인벤토리에 관해서, 접근 통제 시스템, 악성 소프트웨어 감지 시스템 등은 중요한 부분입니다. 더 중요한 문제는 접근 통제의 범위를 어디까지 확대로 확장할 것인지 하는 것입니다.

4. **데이터 보안**: 데이터의 유효성, 접근 가능성을 위한 통제, 그리고 데이터의 중요성을 이해하며 보안 전략을 만드는 것입니다.

5. **개인 정보 보호**: 개인 정보가 침해되는 경우 대처 방법, 그리고 개인 정보의 수집, 보유, 공유 방침 등이 중요한 이슈입니다. 

6. **그리스포인트 보안**: 그린스포인트는 사용자에게 주도적으로 보안을 알리는 것이 중요하다고 지적함을 상기해야 합니다. 

각 회사는 이러한 이슈를 이해하고, 고객과 직원들의 개인정보를 보호하는데 집중해야 합니다. 새로운 기술과 시각 속에 항상 관심을 가져야 합니다.


---

## 정리

이 노트북에서 한 일은 이게 전부입니다.

```
messages 목록을 만든다  ->  create() 로 보낸다  ->  choices[0].message.content 를 꺼낸다
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **키는 `.env` 에** | 코드에 적으면 git 에 올라간다 |
| **`messages` 가 전부** | 모델은 우리가 보낸 목록만 본다 |
| **`system` 이 성격을 정한다** | 같은 질문도 역할에 따라 답이 달라진다 |
| **호출은 매번 독립적** | 모델은 지난 대화를 기억하지 못한다 |
| **`usage` 가 요금** | 토큰 단위로 과금된다 |

### 직접 해 보기

1. `system_prompt` 를 "초등학생도 알아듣게 설명하세요"로 바꿔 보세요. 답이 어떻게 달라지나요?
2. `system_prompt` 를 아예 빼고 `user` 만 보내 보세요. 무엇이 달라지나요?
3. Step 3 의 "홍길동" 메시지와 `assistant` 답변을 `messages` 에 **같이 넣고**
   "내 이름이 뭐라고 했지?" 라고 물어 보세요. 이번엔 기억할 것입니다.

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_2_gemini` · `1_3_ollama` | 같은 일을 다른 모델로 — 코드가 얼마나 비슷한지 |
| `1_5_tokenizer` | 토큰이 정확히 무엇인가 |
| `1_6_stateless` | 모델이 기억하지 못한다는 것의 의미와 해결 |